In [1]:
import json, glob, re, pycm, pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt, scipy.stats as stats
from IPython.display import display, Markdown

In [162]:
RUN_VERSION = "v29"

In [ ]:
data = []
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    df = pd.DataFrame(json.load(open(file, 'r')))
    keys = {
        "Judge Model": model,
        "Prompt": prompt,
        "Dataset": dataset
    }
    n = len(df)
    wk_v_counts = df["wk_v"].value_counts()
    wk_v_freq = (wk_v_counts / n).to_dict()
    I_freq = {
        '<t,t>': 0.0, 
        '<t,e>': 0.0, 
        '<t,f>': 0.0, 
        '<e,t>': 0.0, 
        '<e,e>': 0.0, 
        '<e,f>': 0.0, 
        '<f,t>': 0.0, 
        '<f,e>': 0.0, 
        '<f,f>': 0.0
    }
    I_counts = df["I"].value_counts()
    I_freq_rec = (I_counts / n).to_dict()
    for tv in I_freq_rec:
        I_freq[tv] = I_freq_rec[tv]
    addl_stats = { 
        'avg execution time': df['execution_time'].mean(),
        'std execution time': df['execution_time'].std(),
        'avg tokens used': df['tokens_used'].mean(),
        'std tokens used': df['tokens_used'].std(),
        'coverage': (n - wk_v_counts['e']) / n
    }
    data.append({ **keys, **wk_v_freq, **I_freq, **addl_stats })
df1 = pd.DataFrame.from_records(data)
df1 = df1.round(3)
df1 = df1.sort_values(["Judge Model", "Dataset", "Prompt"])

In [249]:
df1

,Judge Model,Prompt,Dataset,e,f,t,"<t,t>","<t,e>","<t,f>","<e,t>","<e,e>","<e,f>","<f,t>","<f,e>","<f,f>",avg execution time,std execution time,avg tokens used,std tokens used,coverage
33,claude-3-5-haiku-20241022,baseline,gpqa,0.16,0.25,0.59,0.08,0.0,0.59,0.0,0.0,0.0,0.25,0.0,0.08,31.859,4.485,2642.59,648.881,0.84
14,claude-3-5-haiku-20241022,few,gpqa,0.53,0.19,0.28,0.48,0.0,0.28,0.0,0.0,0.0,0.19,0.0,0.05,46.959,4.305,7656.06,648.157,0.47
19,claude-3-5-haiku-20241022,zero,gpqa,0.58,0.19,0.23,0.50,0.0,0.23,0.0,0.0,0.0,0.19,0.0,0.08,43.194,3.771,4216.20,660.958,0.42
34,claude-3-5-haiku-20241022,baseline,simpleqa,0.52,0.18,0.30,0.01,0.0,0.30,0.0,0.0,0.0,0.18,0.0,0.51,15.979,3.291,1009.33,163.143,0.48
26,claude-3-5-haiku-20241022,few,simpleqa,0.61,0.21,0.18,0.14,0.0,0.18,0.0,0.0,0.0,0.21,0.0,0.47,41.053,4.229,6438.93,179.820,0.39
31,claude-3-5-haiku-20241022,zero,simpleqa,0.69,0.20,0.11,0.18,0.0,0.11,0.0,0.0,0.0,0.20,0.0,0.51,39.042,3.166,3107.98,143.984,0.31
24,claude-3-5-sonnet-20241022,baseline,gpqa,0.23,0.34,0.43,0.19,0.0,0.43,0.0,0.0,0.0,0.34,0.0,0.04,39.542,6.273,2965.82,765.463,0.77
4,claude-3-5-sonnet-20241022,few,gpqa,0.44,0.33,0.23,0.43,0.0,0.23,0.0,0.0,0.0,0.33,0.0,0.01,60.267,5.255,8043.21,702.081,0.56
17,claude-3-5-sonnet-20241022,zero,gpqa,0.49,0.30,0.21,0.45,0.0,0.21,0.0,0.0,0.0,0.30,0.0,0.04,61.849,5.505,4848.79,704.905,0.51
18,claude-3-5-sonnet-20241022,baseline,simpleqa,0.47,0.30,0.23,0.04,0.0,0.23,0.0,0.0,0.0,0.30,0.0,0.43,23.781,5.082,1197.56,187.527,0.53


In [ ]:
cms = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms:
        cms[model] = {}
    if prompt not in cms[model]:
        cms[model][prompt] = {}
    df = pd.DataFrame.from_records(json.load(open(file, 'r')))
    cms[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v"].tolist(), digit=2, classes=[ 't', 'f' ])

data = [
    [ 
        model, 
        prompt, 
        dataset, 
        cms[model][prompt][dataset].F1_Macro, 
        cms[model][prompt][dataset].ACC_Macro, 
        cms[model][prompt][dataset].FPR['t'], 
        cms[model][prompt][dataset].FNR['t'], 
        cms[model][prompt][dataset].F1['t'], 
        cms[model][prompt][dataset].F1['f']
    ] 
    for model in cms 
    for prompt in cms[model] 
    for dataset in cms[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Macro-F1", "Acc.", "FPR", "FNR", "F1 (+)", "F1 (-)"]
df2 = pd.DataFrame(data, columns=column_names)
df2 = df2.round(3)
df2 = df2.sort_values(["Judge Model", "Dataset", "Prompt"])

In [234]:
df2

,Judge Model,Prompt,Dataset,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-)
34,claude-3-5-haiku-20241022,baseline,gpqa,0.562,0.571,0.617,0.189,0.625,0.500
30,claude-3-5-haiku-20241022,few,gpqa,0.638,0.638,0.464,0.211,0.638,0.638
32,claude-3-5-haiku-20241022,zero,gpqa,0.660,0.667,0.414,0.154,0.611,0.708
35,claude-3-5-haiku-20241022,baseline,simpleqa,0.666,0.667,0.464,0.150,0.680,0.652
31,claude-3-5-haiku-20241022,few,simpleqa,0.606,0.615,0.375,0.400,0.545,0.667
33,claude-3-5-haiku-20241022,zero,simpleqa,0.689,0.710,0.211,0.417,0.609,0.769
23,claude-3-5-sonnet-20241022,baseline,gpqa,0.701,0.701,0.366,0.222,0.709,0.693
18,claude-3-5-sonnet-20241022,few,gpqa,0.708,0.714,0.226,0.360,0.667,0.750
20,claude-3-5-sonnet-20241022,zero,gpqa,0.717,0.725,0.233,0.333,0.667,0.767
22,claude-3-5-sonnet-20241022,baseline,simpleqa,0.866,0.868,0.103,0.167,0.851,0.881


In [250]:
df = df2.merge(df1, on=['Judge Model', 'Prompt', 'Dataset'], how='inner')

In [251]:
df

,Judge Model,Prompt,Dataset,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-),e,...,"<e,e>","<e,f>","<f,t>","<f,e>","<f,f>",avg execution time,std execution time,avg tokens used,std tokens used,coverage
0,claude-3-5-haiku-20241022,baseline,gpqa,0.562,0.571,0.617,0.189,0.625,0.500,0.16,...,0.0,0.0,0.25,0.0,0.08,31.859,4.485,2642.59,648.881,0.84
1,claude-3-5-haiku-20241022,few,gpqa,0.638,0.638,0.464,0.211,0.638,0.638,0.53,...,0.0,0.0,0.19,0.0,0.05,46.959,4.305,7656.06,648.157,0.47
2,claude-3-5-haiku-20241022,zero,gpqa,0.660,0.667,0.414,0.154,0.611,0.708,0.58,...,0.0,0.0,0.19,0.0,0.08,43.194,3.771,4216.20,660.958,0.42
3,claude-3-5-haiku-20241022,baseline,simpleqa,0.666,0.667,0.464,0.150,0.680,0.652,0.52,...,0.0,0.0,0.18,0.0,0.51,15.979,3.291,1009.33,163.143,0.48
4,claude-3-5-haiku-20241022,few,simpleqa,0.606,0.615,0.375,0.400,0.545,0.667,0.61,...,0.0,0.0,0.21,0.0,0.47,41.053,4.229,6438.93,179.820,0.39
5,claude-3-5-haiku-20241022,zero,simpleqa,0.689,0.710,0.211,0.417,0.609,0.769,0.69,...,0.0,0.0,0.20,0.0,0.51,39.042,3.166,3107.98,143.984,0.31
6,claude-3-5-sonnet-20241022,baseline,gpqa,0.701,0.701,0.366,0.222,0.709,0.693,0.23,...,0.0,0.0,0.34,0.0,0.04,39.542,6.273,2965.82,765.463,0.77
7,claude-3-5-sonnet-20241022,few,gpqa,0.708,0.714,0.226,0.360,0.667,0.750,0.44,...,0.0,0.0,0.33,0.0,0.01,60.267,5.255,8043.21,702.081,0.56
8,claude-3-5-sonnet-20241022,zero,gpqa,0.717,0.725,0.233,0.333,0.667,0.767,0.49,...,0.0,0.0,0.30,0.0,0.04,61.849,5.505,4848.79,704.905,0.51
9,claude-3-5-sonnet-20241022,baseline,simpleqa,0.866,0.868,0.103,0.167,0.851,0.881,0.47,...,0.0,0.0,0.30,0.0,0.43,23.781,5.082,1197.56,187.527,0.53


In [226]:
latex = df2.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:0.3g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Evaluation of different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
    label="tab:expresults",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(latex)

\begin{table}[htbp]
\caption{Evaluation of different judge models and evaluation prompts on three datasets (N=100).}
\label{tab:expresults}
\begin{tabular}{lllrrrrrr}
\toprule
Judge Model & Prompt & Dataset & Macro-F1 & Acc. & FPR & FNR & F1 (+) & F1 (-) \\
\midrule
claude-3-5-haiku-20241022 & baseline & gpqa & 0.772 & 0.8 & 0 & 0.471 & 0.692 & 0.852 \\
claude-3-5-haiku-20241022 & few & gpqa & 0.772 & 0.8 & 0 & 0.471 & 0.692 & 0.852 \\
claude-3-5-haiku-20241022 & zero & gpqa & 0.772 & 0.8 & 0 & 0.471 & 0.692 & 0.852 \\
claude-3-5-haiku-20241022 & baseline & simpleqa & 0.772 & 0.8 & 0 & 0.471 & 0.692 & 0.852 \\
claude-3-5-haiku-20241022 & few & simpleqa & 0.772 & 0.8 & 0 & 0.471 & 0.692 & 0.852 \\
claude-3-5-haiku-20241022 & zero & simpleqa & 0.772 & 0.8 & 0 & 0.471 & 0.692 & 0.852 \\
claude-3-5-sonnet-20241022 & baseline & gpqa & 0.772 & 0.8 & 0 & 0.471 & 0.692 & 0.852 \\
claude-3-5-sonnet-20241022 & few & gpqa & 0.772 & 0.8 & 0 & 0.471 & 0.692 & 0.852 \\
claude-3-5-sonnet-20241022 & z